# 01 - Deploy Foundry, PostgreSQL, and the public LiteLLM proxy

This notebook creates the Foundry model, a lowest-cost PostgreSQL Flexible Server for LiteLLM key authentication, builds the LiteLLM image in ACR, and deploys an external HTTPS Container App.

The proxy is disabled during the first Bicep pass so the Foundry API key can be retrieved and injected as a secret during the second pass. PostgreSQL is deployed in the first pass because the current LiteLLM image needs a database to validate bearer keys on `/v1/messages`.

**Before deploying:** validate `MODEL_NAME` and `MODEL_VERSION` in the selected region. GPT-5.4 availability and version strings can change.

**Cost note:** PostgreSQL uses the cheapest practical prototype settings: Burstable `Standard_B1ms`, 32 GiB storage, no high availability, seven-day backup retention, and geo-redundant backup disabled.

In [ ]:
import json
import os
import secrets
import subprocess
import sys
from pathlib import Path

RESOURCE_GROUP = 'rg-litellm01-foundry-claude'
LOCATION = 'eastus2'
DATABASE_LOCATION = 'centralus'
DEMO_ID = 'litellm01'
MODEL_NAME = 'gpt-5.4'
MODEL_VERSION = '2026-03-05'
DEPLOYMENT_NAME = 'gpt-5.4'
DEPLOYMENT_NAME_ALIAS = 'azure-gpt-5.4'
DEPLOYMENT_NAME_FOR_BICEP = DEPLOYMENT_NAME
DATABASE_PASSWORD = 'Pg-' + secrets.token_urlsafe(24)

candidate_roots = [Path.cwd(), *Path.cwd().parents]
demo_root = next((root / 'demos' / 'litellm01-foundry-claude' for root in candidate_roots if (root / 'demos' / 'litellm01-foundry-claude').is_dir()), None)
if demo_root is None:
    raise RuntimeError('Could not locate demos/litellm01-foundry-claude from the notebook working directory.')
demo_root = demo_root.resolve()
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError('Azure CLI not found. Run 00_setup.ipynb and ensure az is on PATH.')
print(f'Using Azure CLI: {az_cmd}')
print(f'Resource group: {RESOURCE_GROUP}')
print(f'Database location: {DATABASE_LOCATION}')
print(f'Foundry model: {MODEL_NAME} / {MODEL_VERSION}')

## Step 1 - Deploy Foundry, ACR, Container Apps base resources, and PostgreSQL

This first pass deploys the resources that do not depend on the LiteLLM image or Foundry API key:

- Foundry AI Services account, project, and GPT-5.4 model deployment
- Azure Container Registry
- User-assigned managed identity with `AcrPull` on the registry
- Log Analytics workspace and Container Apps environment
- PostgreSQL Flexible Server plus the `litellm` database

The PostgreSQL administrator password is generated in the setup cell and passed as a secure Bicep parameter. It is not printed and is not written to `env/.env`.

In [ ]:
def _sanitize_arg(arg):
    if arg.startswith('azureApiKey='):
        return 'azureApiKey=<redacted>'
    if arg.startswith('proxyMasterKey='):
        return 'proxyMasterKey=<redacted>'
    if arg.startswith('databasePassword='):
        return 'databasePassword=<redacted>'
    return arg

def _format_az_error(args, result):
    command_text = ' '.join(_sanitize_arg(a) for a in args)
    return (
        f"az {command_text} failed (exit {result.returncode})\n"
        f"stderr: {result.stderr.strip()}\n"
        f"stdout: {result.stdout.strip()}"
    )

def run_az(args, allow_failure=False):
    result = subprocess.run([az_cmd, *args], capture_output=True, text=True)
    if result.returncode != 0 and not allow_failure:
        raise RuntimeError(_format_az_error(args, result))
    return result

template_file = str(demo_root / 'infra' / 'main.bicep')
run_az(['group', 'create', '--name', RESOURCE_GROUP, '--location', LOCATION, '--tags', 'SecurityControl=Ignore', '-o', 'none'])
run_az([
    'deployment', 'group', 'create', '--resource-group', RESOURCE_GROUP,
    '--name', 'litellm01-bootstrap', '--template-file', template_file,
    '--parameters', f'location={LOCATION}', f'databaseLocation={DATABASE_LOCATION}', f'demoId={DEMO_ID}',
    f'deploymentName={DEPLOYMENT_NAME_FOR_BICEP}', f'modelName={MODEL_NAME}',
    f'modelVersion={MODEL_VERSION}', f'databasePassword={DATABASE_PASSWORD}'
])
print('Bootstrap deployment completed. The public proxy is not enabled yet.')

## Step 2 - Retrieve Foundry credentials and build the LiteLLM image

This step reads the first-pass deployment outputs, retrieves the Foundry account key, and builds the LiteLLM proxy image in ACR. The image includes the demo's `config.yaml`, which maps the public `azure-gpt-5.4` alias to the Foundry deployment.

In [ ]:
outputs = json.loads(run_az(['deployment', 'group', 'show', '-g', RESOURCE_GROUP, '-n', 'litellm01-bootstrap', '--query', 'properties.outputs', '-o', 'json']).stdout)
def output(name):
    return (outputs.get(name) or {}).get('value', '')
ai_account_name = output('aiServicesAccountName')
acr_name = output('acrName')
acr_login_server = output('acrLoginServer')
ai_endpoint = output('aiServicesEndpoint')
if not ai_account_name or not acr_name:
    raise RuntimeError(f'Expected deployment outputs were missing: {list(outputs)}')

api_key_result = run_az(['cognitiveservices', 'account', 'keys', 'list', '--resource-group', RESOURCE_GROUP, '--name', ai_account_name, '--query', 'key1', '-o', 'tsv'])
azure_api_key = api_key_result.stdout.strip()
if not azure_api_key:
    raise RuntimeError('Foundry account key was empty. Confirm local auth is enabled on the AIServices account.')

image_name = 'litellm-proxy'
image_tag = 'latest'
image_reference = f'{acr_login_server}/{image_name}:{image_tag}'
build = run_az(['acr', 'build', '--registry', acr_name, '--image', f'{image_name}:{image_tag}', str(demo_root / 'app')])
print(build.stdout)
print(f'Image built: {image_reference}')

## Step 3 - Enable the public proxy and write the connection details

This second Bicep pass creates or updates the public Container App using the built LiteLLM image. It injects three secrets into the app:

- Foundry API key
- LiteLLM master key
- PostgreSQL `DATABASE_URL`

The notebook reuses `LITELLM_MASTER_KEY` from `env/.env` when present to keep admin auth stable across reruns. It then generates a fresh database-backed client token and writes client-facing values to `env/.env`.

In [ ]:
import json
import secrets
import time
import urllib.error
import urllib.request

env_path = demo_root / 'env' / '.env'
existing_env = {}
if env_path.exists():
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        existing_env[key] = value

proxy_master_key = existing_env.get('LITELLM_MASTER_KEY', '').strip()
if proxy_master_key:
    print('Reusing existing LiteLLM master key from env/.env to keep admin auth stable across reruns.')
else:
    proxy_master_key = 'sk-' + secrets.token_urlsafe(32)
    print('Generated new LiteLLM master key (first run or env reset).')

if 'DATABASE_LOCATION' not in globals():
    DATABASE_LOCATION = 'centralus'
if 'DATABASE_PASSWORD' not in globals():
    DATABASE_PASSWORD = 'Pg-' + secrets.token_urlsafe(24)
    print('Generated a new database password for the existing PostgreSQL server and proxy update.')
template_file = str(demo_root / 'infra' / 'main.bicep')

if 'run_az' not in globals():
    raise RuntimeError('Run the setup and Step 1 cells first so Azure CLI helpers are initialized.')

if 'ai_account_name' not in globals() or 'acr_name' not in globals():
    bootstrap_outputs = json.loads(run_az([
        'deployment', 'group', 'show', '-g', RESOURCE_GROUP, '-n', 'litellm01-bootstrap',
        '--query', 'properties.outputs', '-o', 'json'
    ]).stdout)
    def output(name):
        return (bootstrap_outputs.get(name) or {}).get('value', '')
    ai_account_name = output('aiServicesAccountName')
    acr_name = output('acrName')
    acr_login_server = output('acrLoginServer')
    ai_endpoint = output('aiServicesEndpoint')
    if not ai_account_name or not acr_name:
        raise RuntimeError(f'Bootstrap deployment outputs were missing: {list(bootstrap_outputs)}')

if 'azure_api_key' not in globals():
    azure_api_key = run_az([
        'cognitiveservices', 'account', 'keys', 'list',
        '--resource-group', RESOURCE_GROUP, '--name', ai_account_name,
        '--query', 'key1', '-o', 'tsv'
    ]).stdout.strip()
    if not azure_api_key:
        raise RuntimeError('Foundry account key was empty. Confirm local auth is enabled.')

if 'image_reference' not in globals():
    image_reference = f'{acr_login_server}/litellm-proxy:latest'
    print(f'Reconstructed image reference: {image_reference}')

deploy_args = [
    'deployment', 'group', 'create', '--resource-group', RESOURCE_GROUP,
    '--name', 'litellm01-proxy', '--template-file', template_file,
    '--parameters', f'location={LOCATION}', f'databaseLocation={DATABASE_LOCATION}', f'demoId={DEMO_ID}',
    f'deploymentName={DEPLOYMENT_NAME_FOR_BICEP}', f'modelName={MODEL_NAME}',
    f'modelVersion={MODEL_VERSION}', f'containerImageReference={image_reference}',
    f'azureApiKey={azure_api_key}', f'proxyMasterKey={proxy_master_key}',
    f'databasePassword={DATABASE_PASSWORD}'
]

run_az(deploy_args)

proxy_outputs = json.loads(run_az([
    'deployment', 'group', 'show', '-g', RESOURCE_GROUP, '-n', 'litellm01-proxy',
    '--query', 'properties.outputs', '-o', 'json'
    ]).stdout)
proxy_url = (proxy_outputs.get('proxyUrl') or {}).get('value', '')
if not proxy_url:
    raise RuntimeError('Proxy URL was empty. Inspect the litellm01-proxy deployment operations.')

# Generate a database-backed client token for normal API traffic.
keygen_payload = {
    'duration': '7d',
    'models': [DEPLOYMENT_NAME_ALIAS],
    'max_budget': 500,
    'spend': 0,
}

proxy_client_key = ''
last_keygen_error = ''
for attempt in range(1, 21):
    keygen_request = urllib.request.Request(
        url=f'{proxy_url}/key/generate',
        data=json.dumps(keygen_payload).encode('utf-8'),
        headers={'Authorization': f'Bearer {proxy_master_key}', 'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with urllib.request.urlopen(keygen_request, timeout=60) as response:
            keygen_status = response.getcode()
            keygen_body = response.read().decode('utf-8')
    except urllib.error.HTTPError as exc:
        keygen_status = exc.code
        keygen_body = exc.read().decode('utf-8', errors='replace')
    except urllib.error.URLError as exc:
        keygen_status = None
        keygen_body = f'URLError: {exc}'

    if keygen_status is not None and 200 <= keygen_status < 300:
        proxy_client_key = (json.loads(keygen_body) or {}).get('key', '').strip()
        if proxy_client_key:
            break
        last_keygen_error = 'Key generation returned 2xx but no key in payload.'
    else:
        last_keygen_error = f'Attempt {attempt}: status={keygen_status}, body={keygen_body[:500]}'

    is_token_race = (
        keygen_status == 401 and 'token_not_found_in_db' in keygen_body
    )
    if not is_token_race or attempt == 20:
        break
    print(f'Key generation not ready yet (attempt {attempt}/20). Waiting 6s before retry...')
    time.sleep(6)

if not proxy_client_key:
    raise RuntimeError(f'Failed to generate proxy client key. Last error: {last_keygen_error}')

env_path.parent.mkdir(parents=True, exist_ok=True)
env_path.write_text('\n'.join([
    f'AZURE_SUBSCRIPTION_ID={run_az(["account", "show", "--query", "id", "-o", "tsv"]).stdout.strip()}',
    f'AZURE_RESOURCE_GROUP={RESOURCE_GROUP}',
    f'AZURE_AI_ENDPOINT={ai_endpoint}',
    f'AZURE_AI_ACCOUNT_NAME={ai_account_name}',
    f'AZURE_MODEL_DEPLOYMENT={DEPLOYMENT_NAME_FOR_BICEP}',
    f'LITELLM_MODEL_ALIAS={DEPLOYMENT_NAME_ALIAS}',
    f'LITELLM_PROXY_URL={proxy_url}',
    f'ANTHROPIC_BASE_URL={proxy_url}',
    f'LITELLM_MASTER_KEY={proxy_master_key}',
    f'LITELLM_CLIENT_KEY={proxy_client_key}',
    f'ANTHROPIC_AUTH_TOKEN={proxy_client_key}',
    'CLAUDE_CODE_ENABLE_GATEWAY_MODEL_DISCOVERY=1',
    f'CLAUDE_CODE_MODEL={DEPLOYMENT_NAME_ALIAS}',
]) + '\n', encoding='utf-8')
print(f'Public LiteLLM URL: {proxy_url}')
print(f'Claude Code model: {DEPLOYMENT_NAME_ALIAS}')
print(f'Master key: {proxy_master_key[:8]}... (saved to env/.env)')
print(f'Client key: {proxy_client_key[:8]}... (saved to env/.env)')
print(f'Connection details written to {env_path}')